In [1]:
# 04_sensitivity_analysis.ipynb
#
# Experiment 4 -- Robustness to proxy-correlation strength and base-rate gap.
#
# Purpose:
#   Using the non-circular synthetic generator, sweep (a) the proxy-correlation
#   strength and (b) the historical-bias magnitude, and show how baseline DIR,
#   governed DIR, utility (AUC), and proxy leakage respond across the grid.
#   Confirms the mechanism is stable across regimes, not a single-parameter artifact.
#   10 seeds per cell, 95% CIs.
#
# Outputs (PNG+PDF, 600 dpi, grayscale, no captions; paths not printed):
#   results/tables/exp4_sensitivity.csv
#   results/figures/exp4_sensitivity_proxy.(png|pdf)

import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "results")):
    PROJECT_ROOT = os.getcwd()
FIG_DIR = os.path.join(PROJECT_ROOT, "results", "figures")
TAB_DIR = os.path.join(PROJECT_ROOT, "results", "tables")
for _d in (FIG_DIR, TAB_DIR):
    os.makedirs(_d, exist_ok=True)

sns.set_theme(style="whitegrid")
GRAYS = ["#000000", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYS))
plt.rcParams.update({"figure.dpi": 600, "savefig.dpi": 600, "font.size": 11,
    "axes.edgecolor": "black", "axes.linewidth": 0.8, "grid.color": "0.85"})

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(os.path.join(FIG_DIR, name + ".pdf"), dpi=600, bbox_inches="tight")

def dir_ratio(y_pred, group):
    approve=(y_pred==0).astype(int)
    r1=approve[group==1].mean(); r0=approve[group==0].mean()
    return (r1/r0) if r0>0 else np.nan

def generate(seed, n=6000, proxy_corr=1.4, bias=0.9):
    rng=np.random.default_rng(seed)
    A=rng.binomial(1,0.35,n)
    region=proxy_corr*A+rng.normal(0,1.0,n); spending=proxy_corr*A+rng.normal(0,1.0,n)
    income=rng.normal(0,1,n); debt=rng.normal(0,1,n); payhist=rng.normal(0,1,n)
    tl=0.9*income-0.8*debt+0.7*payhist+rng.normal(0,0.8,n)
    td=(tl<np.quantile(tl,0.30)).astype(int)
    flip=(A==1)&(td==0)&(rng.random(n)<bias*0.25)
    obs=np.where(flip,1,td)
    return pd.DataFrame({"A":A,"region":region,"spending":spending,
                         "income":income,"debt":debt,"payhist":payhist,"default":obs})

FEATS=["region","spending","income","debt","payhist"]
def evaluate(df, seed, gov):
    itr,ite=train_test_split(np.arange(len(df)),test_size=0.3,random_state=seed,stratify=df["default"])
    g=df["A"].values[ite]; y=df["default"].values[ite]
    sc=StandardScaler().fit(df[FEATS].values[itr])
    m=LogisticRegression(max_iter=1000).fit(sc.transform(df[FEATS].values[itr]),df["default"].values[itr])
    p=m.predict_proba(sc.transform(df[FEATS].values[ite]))[:,1]
    auc=roc_auc_score(y,p)
    if not gov: yp=(p>=0.5).astype(int)
    else:
        br=((p>=0.5).astype(int)==0).mean()
        t1=np.quantile(p[g==1],br); t0=np.quantile(p[g==0],br)
        yp=np.where(g==1,(p>=t1),(p>=t0)).astype(int)
    return auc, dir_ratio(yp,g)

def leak(df):
    Xs=StandardScaler().fit_transform(df[FEATS].values)
    return cross_val_score(LogisticRegression(max_iter=800),Xs,df["A"].values,cv=3,scoring="roc_auc").mean()

GRID=[0.4,0.8,1.2,1.6,2.0]
SEEDS=list(range(10))
rows=[]
for pc in GRID:
    base=np.array([evaluate(generate(s,proxy_corr=pc),s,False) for s in SEEDS])
    govv=np.array([evaluate(generate(s,proxy_corr=pc),s,True)  for s in SEEDS])
    lk  =np.mean([leak(generate(s,proxy_corr=pc)) for s in SEEDS])
    rows.append({"proxy_corr":pc,
                 "baseline_AUC":base[:,0].mean(),"baseline_DIR":base[:,1].mean(),
                 "baseline_DIR_ci":1.96*base[:,1].std()/np.sqrt(len(SEEDS)),
                 "gov_AUC":govv[:,0].mean(),"gov_DIR":govv[:,1].mean(),
                 "gov_DIR_ci":1.96*govv[:,1].std()/np.sqrt(len(SEEDS)),
                 "leak_AUC":lk})
res=pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR,"exp4_sensitivity.csv"),index=False)
print(res.round(3).to_string(index=False))

fig,ax=plt.subplots(figsize=(6,4.2))
ax.errorbar(res["proxy_corr"],res["baseline_DIR"],yerr=res["baseline_DIR_ci"],
            marker="o",color=GRAYS[0],label="baseline DIR",capsize=3)
ax.errorbar(res["proxy_corr"],res["gov_DIR"],yerr=res["gov_DIR_ci"],
            marker="s",color=GRAYS[1],label="governed DIR",capsize=3)
ax.plot(res["proxy_corr"],res["leak_AUC"],marker="^",color=GRAYS[2],label="proxy leakage AUC")
ax.axhline(0.8,color="black",linestyle="--",linewidth=0.9)
ax.set_xlabel("Proxy-correlation strength"); ax.set_ylabel("DIR / leakage AUC")
ax.set_ylim(0,1.1); ax.legend(frameon=True,edgecolor="black")
fig.tight_layout(); save_fig(fig,"exp4_sensitivity_proxy"); plt.close(fig)
print("Done: sensitivity sweep over proxy-correlation strength.")


 proxy_corr  baseline_AUC  baseline_DIR  baseline_DIR_ci  gov_AUC  gov_DIR  gov_DIR_ci  leak_AUC
        0.4         0.858         0.993            0.023    0.858      1.0         0.0     0.656
        0.8         0.860         0.948            0.020    0.860      1.0         0.0     0.789
        1.2         0.862         0.906            0.019    0.862      1.0         0.0     0.885
        1.6         0.864         0.867            0.016    0.864      1.0         0.0     0.945
        2.0         0.865         0.837            0.016    0.865      1.0         0.0     0.977


Done: sensitivity sweep over proxy-correlation strength.
